In [2]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from tensorflow.keras.layers import *
from tensorflow.keras.models import Model

print("Imported Successfully")

ModuleNotFoundError: No module named 'tensorflow'

In [3]:
%pip install tf-nightly

ERROR: Could not find a version that satisfies the requirement tf-nightly (from versions: none)
ERROR: No matching distribution found for tf-nightly
Note: you may need to restart the kernel to use updated packages.


In [ ]:
images = []
masks = []

image_path = "../data/segmentation/images"
mask_path = "../data/segmentation/masks"

image_files = sorted(os.listdir(image_path))
mask_files = sorted(os.listdir(mask_path))

for img_file, mask_file in zip(image_files, mask_files):

    img = cv2.imread(os.path.join(image_path, img_file))
    img = cv2.resize(img, (128,128))

    mask = cv2.imread(
        os.path.join(mask_path, mask_file),
        cv2.IMREAD_GRAYSCALE
    )

    mask = cv2.resize(mask, (128,128))

    images.append(img)
    masks.append(mask)

X = np.array(images) / 255.0

y = np.array(masks) / 255.0

y = np.expand_dims(y, axis=-1)

print(X.shape)
print(y.shape)

In [ ]:
plt.figure(figsize=(10,5))

for i in range(3):

    plt.subplot(3,2,2*i+1)
    plt.imshow(X[i])
    plt.title("MRI Image")
    plt.axis("off")

    plt.subplot(3,2,2*i+2)
    plt.imshow(y[i].squeeze(), cmap='gray')
    plt.title("Tumour Mask")
    plt.axis("off")

plt.show()

In [ ]:
inputs = Input((128,128,3))

# Encoder
c1 = Conv2D(16, (3,3), activation='relu', padding='same')(inputs)
p1 = MaxPooling2D((2,2))(c1)

c2 = Conv2D(32, (3,3), activation='relu', padding='same')(p1)
p2 = MaxPooling2D((2,2))(c2)

# Bottleneck
b1 = Conv2D(64, (3,3), activation='relu', padding='same')(p2)

# Decoder
u1 = UpSampling2D((2,2))(b1)
c3 = Conv2D(32, (3,3), activation='relu', padding='same')(u1)

u2 = UpSampling2D((2,2))(c3)
c4 = Conv2D(16, (3,3), activation='relu', padding='same')(u2)

outputs = Conv2D(1, (1,1), activation='sigmoid')(c4)

model = Model(inputs, outputs)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=5
)

In [ ]:
prediction = model.predict(
    np.expand_dims(X_test[0], axis=0)
)[0]

In [ ]:
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(X_test[0])
plt.title("Original MRI")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(y_test[0].squeeze(), cmap='gray')
plt.title("True Mask")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(prediction.squeeze(), cmap='gray')
plt.title("Predicted Mask")
plt.axis("off")

plt.show()